# 闭包、装饰器、迭代器与生成器

## 学习目标

理解函数是一等对象、闭包如何保存状态、装饰器如何包装函数，以及生成器为什么能暂停和
恢复。本教程最后会直接读取生成器保存的 frame，让“惰性计算”从一句定义变成可观察状态。

## 准备

使用 `ipywidgets` 选择推进次数，使用 Matplotlib 展示生成器暂停时保存的局部变量。

In [ ]:
%matplotlib inline

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

plt.rcParams["axes.unicode_minus"] = False

In [ ]:
from functools import wraps
from itertools import islice

## 函数对象与闭包

### 1. 函数是一等对象

函数可以赋给变量、作为参数传入，也可以作为返回值。

In [ ]:
def add(left, right):
    return left + right


operation = add
print("函数对象:", operation.__name__)
print("调用结果:", operation(2, 3))

### 2. 闭包保存外层状态

`counter_factory` 返回后，它的局部调用帧虽然结束了，但 `increment` 仍引用其中的
`count` 闭包单元。`nonlocal` 修改的是这份被闭包保留的状态。

In [ ]:
def counter_factory(start=0):
    count = start

    def increment(step=1):
        nonlocal count
        count += step
        return count

    return increment


counter_a = counter_factory()
counter_b = counter_factory(100)
print("A:", counter_a(), counter_a(5))
print("B:", counter_b(), counter_b())

### 3. 装饰器增加横切功能

`@trace` 的核心等价式是 `multiply = trace(multiply)`。调用者最终拿到的是
`wrapper`，`wrapper` 再通过闭包找到原函数。

In [ ]:
def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"调用 {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"返回 {result!r}")
        return result

    return wrapper


@trace
def multiply(left, right):
    return left * right


product = multiply(6, 7)
print("函数名仍被保留:", multiply.__name__)

## 生成器为什么可以暂停

调用生成器函数时只创建生成器对象，函数体尚未执行。每次 `next()` 会从上次暂停位置
继续，运行到下一个 `yield` 后再次暂停。暂停期间，指令位置和局部变量都保存在
`generator.gi_frame` 中。

先运行普通示例，再用下面的滑块观察每一次暂停状态。

In [ ]:
def fibonacci():
    left, right = 0, 1
    while True:
        yield left
        left, right = right, left + right


sequence = fibonacci()
print("生成器对象:", sequence)
print("前 8 项:", list(islice(sequence, 8)))

In [ ]:
def capture_fibonacci_states(step_count=8):
    sequence = fibonacci()
    states = []
    produced = []

    for step in range(1, step_count + 1):
        value = next(sequence)
        produced.append(value)
        frame_locals = dict(sequence.gi_frame.f_locals)
        states.append(
            {
                "step": step,
                "yielded": value,
                "produced": produced.copy(),
                "left": frame_locals["left"],
                "right": frame_locals["right"],
                "line": sequence.gi_frame.f_lineno,
            }
        )
    return states


fibonacci_states = capture_fibonacci_states()


def draw_generator_state(step):
    state = fibonacci_states[step - 1]
    figure, axis = plt.subplots(figsize=(10, 4.6))
    axis.set_xlim(0, 10)
    axis.set_ylim(0, 4)
    axis.axis("off")
    axis.set_title(
        f"After next() #{step}: suspended at yield",
        fontsize=15,
    )

    boxes = [
        (1.7, 2.2, "values produced", repr(state["produced"]), "#dbeafe"),
        (
            5.0,
            2.2,
            "saved local state",
            f"left = {state['left']}\nright = {state['right']}",
            "#fef3c7",
        ),
        (
            8.3,
            2.2,
            "next next() call",
            "resume after yield\nupdate left, right",
            "#dcfce7",
        ),
    ]
    for x, y, title, detail, color in boxes:
        axis.text(
            x,
            y,
            f"{title}\n\n{detail}",
            ha="center",
            va="center",
            fontsize=11,
            bbox={
                "boxstyle": "round,pad=0.6",
                "facecolor": color,
                "edgecolor": "#334155",
            },
        )

    for start, end in [((2.8, 2.2), (3.8, 2.2)), ((6.2, 2.2), (7.1, 2.2))]:
        axis.annotate(
            "",
            xy=end,
            xytext=start,
            arrowprops={"arrowstyle": "->", "lw": 2, "color": "#475569"},
        )

    axis.text(
        5,
        0.45,
        (
            f"yielded {state['yielded']} at source line {state['line']}; "
            "local variables remain alive while suspended."
        ),
        ha="center",
        fontsize=10,
        color="#0f172a",
    )
    plt.show()
    plt.close(figure)


generator_step = widgets.IntSlider(
    value=1,
    min=1,
    max=len(fibonacci_states),
    step=1,
    description="next 次数：",
    continuous_update=False,
    style={"description_width": "initial"},
)
generator_output = widgets.Output()


def render_generator_step(_change=None):
    with generator_output:
        clear_output(wait=True)
        draw_generator_state(generator_step.value)


generator_step.observe(render_generator_step, names="value")
display(widgets.VBox([generator_step, generator_output]))
render_generator_step()

滑块前进时，`produced` 逐渐增长；图中央的 `left` 和 `right` 是生成器暂停时真实保存的
局部变量。下一次 `next()` 不会从函数开头重跑，而是从当前 `yield` 之后恢复。

## 自定义迭代器协议

In [ ]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current == 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


print("倒计时:", list(Countdown(5)))

## 常见误解

- “生成器保存所有结果”：它主要保存执行状态，已经交给调用者的值不会自动组成列表。
- “调用生成器函数会立刻执行”：调用只返回生成器对象，第一次 `next()` 才进入函数体。
- “可迭代对象就是迭代器”：可迭代对象能产生迭代器；迭代器还要保存当前位置并实现
  `__next__`。
- “装饰器只是语法”：装饰器会在函数定义阶段替换绑定到函数名上的对象。

## 面试时可以这样解释

> 生成器是保存了指令位置和局部变量的可恢复执行帧。`next()` 从上次 `yield` 后恢复，
> 运行到下一个 `yield` 再暂停，因此可以按需生成数据，而不必提前构造完整结果。

## 继续探索

1. 给 `trace` 增加耗时统计。
2. 写一个可以重复迭代的 `CountdownIterable`，让 `__iter__` 返回新的迭代器。
3. 比较列表推导式和生成器表达式的内存占用。